# S10 · Serie de tiempo: velocidad del viento **dentro de 1 hora** + inferencia en streaming
**Nayder Arce Mayta** · histórico horario de Juliaca (U1) → modelo → aplicado en vivo a `viento-eventos`

Reglas de la sesión: baseline de **persistencia** primero, split **cronológico**, ≥ 2 algoritmos, guardar
`Pipeline` completo, inferencia sin ventanas ni estado, `foreachBatch` → Parquet con checkpoint.

In [1]:
import os, time, json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, IntegerType

BASE = os.getenv("VIENTO_BASE", "/opt")
ART = f"{BASE}/artifacts"
KAFKA = os.getenv("KAFKA_BOOTSTRAP", "kafka:9092")
TOPIC = "viento-eventos"
PUSHGATEWAY = os.getenv("PUSHGATEWAY", "http://pushgateway:9091")

spark = (SparkSession.builder
         .appName("s10-series-tiempo-viento-nayder")
         .master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
         .config("spark.sql.shuffle.partitions", "3")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.ui.port", "4040")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "|", time.strftime("%Y-%m-%d %H:%M:%S"))

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-73ddd243-3d58-491e-ac24-8b7b48c5ac2b;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Spark 4.2.0 | 2026-11-03 01:19:39


## 1. Dataset supervisado: label = velocidad de la hora siguiente (`lead`)

In [2]:
from pyspark.sql.window import Window
from pyspark.sql.types import StructField, StructType as ST
CSV_HORARIO = f"{BASE}/data/viento_juliaca_horario.csv"   # lo descarga el notebook de la U1
esquema_csv = ST([StructField(c_, StringType() if c_ == "fecha_hora" else DoubleType(), True) for c_ in
                  ["fecha_hora", "temperatura", "humedad", "presion", "nubosidad", "velocidad", "direccion", "rafaga"]])
h = (spark.read.csv(CSV_HORARIO, header=True, schema=esquema_csv)
     .withColumn("ts", F.to_timestamp("fecha_hora", "yyyy-MM-dd'T'HH:mm"))
     .dropna(subset=["velocidad", "rafaga", "direccion", "temperatura", "humedad", "presion"])
     .dropDuplicates(["ts"]))

def features(df):
    """Mismas features en batch y en streaming (clave para no tener training/serving skew)."""
    hora = F.hour("ts_feat") + F.minute("ts_feat") / 60.0
    return (df.withColumn("dir_sin", F.sin(F.radians("direccion"))).withColumn("dir_cos", F.cos(F.radians("direccion")))
              .withColumn("hora_sin", F.sin(2 * 3.141592653589793 * hora / 24))
              .withColumn("hora_cos", F.cos(2 * 3.141592653589793 * hora / 24))
              .withColumn("mes_sin", F.sin(2 * 3.141592653589793 * F.month("ts_feat") / 12))
              .withColumn("mes_cos", F.cos(2 * 3.141592653589793 * F.month("ts_feat") / 12)))

FEATS = ["velocidad", "rafaga", "dir_sin", "dir_cos", "temperatura", "humedad", "presion",
         "hora_sin", "hora_cos", "mes_sin", "mes_cos"]

w = Window.orderBy("ts")
sup = (features(h.withColumn("ts_feat", F.col("ts")))
       .withColumn("label", F.lead("velocidad", 1).over(w))
       .withColumn("ts_sig", F.lead("ts", 1).over(w))
       .filter((F.unix_timestamp("ts_sig") - F.unix_timestamp("ts")) == 3600)   # la hora siguiente debe existir
       .select("ts", *FEATS, "label"))
print("filas supervisadas:", sup.count())

[Stage 0:=============================>                             (1 + 1) / 2]

filas supervisadas: 147455


## 2. Split cronológico 80/20 con `row_number` (nunca `randomSplit`)

In [3]:
idx = sup.withColumn("i", F.row_number().over(Window.orderBy("ts")))
n = idx.count(); corte = int(n * 0.8)
train = idx.filter(F.col("i") <= corte).drop("i").cache()
test = idx.filter(F.col("i") > corte).drop("i").cache()
tmax, tmin = train.agg(F.max("ts")).first()[0], test.agg(F.min("ts")).first()[0]
print(f"train={train.count()} test={test.count()} | max train {tmax} < min test {tmin}: {tmax < tmin}")

train=117964 test=29491 | max train 2023-06-17 08:00:00 < min test 2023-06-17 09:00:00: True


## 3. Baseline de persistencia: velocidad(t+1h) = velocidad(t)

In [4]:
from pyspark.ml.evaluation import RegressionEvaluator
def evaluar(df, pred="prediction"):
    ev = RegressionEvaluator(labelCol="label", predictionCol=pred)
    return {m: round(ev.setMetricName(m).evaluate(df), 4) for m in ["rmse", "mae", "r2"]}
res = {"Baseline persistencia": evaluar(test.withColumn("prediction", F.col("velocidad")))}
res

{'Baseline persistencia': {'rmse': 3.7468, 'mae': 2.5818, 'r2': 0.5273}}

## 4. Modelado comparado: LinearRegression vs RandomForestRegressor

In [5]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
va = VectorAssembler(inputCols=FEATS, outputCol="features")
cands = {"LinearRegression": Pipeline(stages=[va, LinearRegression(labelCol="label")]),
         "RandomForest (50, d8)": Pipeline(stages=[va, RandomForestRegressor(labelCol="label", numTrees=50, maxDepth=8, seed=42)])}
fit = {}
for k, p in cands.items():
    fit[k] = p.fit(train); res[k] = evaluar(fit[k].transform(test))

import pandas as pd
tabla = pd.DataFrame(res).T
b = tabla.loc["Baseline persistencia", "rmse"]
tabla["mejora_rmse_vs_baseline_%"] = ((b - tabla["rmse"]) / b * 100).round(2)
tabla

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
                                                                                

,rmse,mae,r2,mejora_rmse_vs_baseline_%
Baseline persistencia,3.7468,2.5818,0.5273,0.00
LinearRegression,3.3269,2.3410,0.6273,11.21
"RandomForest (50, d8)",3.1649,2.2492,0.6627,15.53


## 5. Selección honesta y despliegue (Pipeline completo)

In [6]:
ganador = tabla.drop("Baseline persistencia")["rmse"].idxmin()
supera = tabla.loc[ganador, "rmse"] < b
print(f"Ganador: {ganador} | RMSE {tabla.loc[ganador,'rmse']} vs baseline {b} → "
      + ("SUPERA a la persistencia" if supera else "NO supera a la persistencia (se reporta tal cual)"))
MODELO = f"{ART}/modelo_viento_1h"
fit[ganador].write().overwrite().save(MODELO)
modelo = PipelineModel.load(MODELO)
modelo.transform(test).select("ts", "velocidad", "label", F.round("prediction", 2).alias("pred_1h")).orderBy(F.desc("ts")).show(5)
json.dump({"ganador": ganador, "supera_baseline": bool(supera), "metricas": tabla.to_dict(orient="index")},
          open(f"{ART}/metricas_s10.json", "w"), indent=2, default=float)

Ganador: RandomForest (50, d8) | RMSE 3.1649 vs baseline 3.7468 → SUPERA a la persistencia


+-------------------+---------+-----+-------+
|                 ts|velocidad|label|pred_1h|
+-------------------+---------+-----+-------+
|2026-10-27 22:00:00|     14.1| 13.4|  12.16|
|2026-10-27 21:00:00|      5.6| 14.1|   6.92|
|2026-10-27 20:00:00|     12.4|  5.6|  10.92|
|2026-10-27 19:00:00|     16.8| 12.4|  15.68|
|2026-10-27 18:00:00|     17.8| 16.8|  14.33|
+-------------------+---------+-----+-------+
only showing top 5 rows


## 6. Inferencia en tiempo real sobre `viento-eventos` (sin ventanas ni estado)

In [7]:
# Contrato viento.lectura v1 (docs/contratos.md)
esquema_viento = (StructType()
    .add("tipoEvento", StringType()).add("version", IntegerType())
    .add("sensorId", StringType()).add("estacion", StringType())
    .add("lat", DoubleType()).add("lon", DoubleType())
    .add("velocidadViento", DoubleType()).add("direccionViento", DoubleType()).add("rafaga", DoubleType())
    .add("temperatura", DoubleType()).add("humedad", DoubleType()).add("presion", DoubleType())
    .add("obsTime", LongType()).add("origen", StringType()).add("timestamp", LongType()))

crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", KAFKA)
         .option("subscribe", TOPIC)
         .option("startingOffsets", "latest")
         .option("failOnDataLoss", "false")
         .load())

eventos = (crudo.select("partition", "offset", F.col("timestamp").alias("kafkaTs"),
                        F.col("value").cast("string").alias("json_str"))
           .select("partition", "offset", "kafkaTs", "json_str",
                   F.from_json("json_str", esquema_viento).alias("e"))
           .select("partition", "offset", "kafkaTs", "json_str", "e.*")
           .withColumn("ts", (F.col("timestamp") / 1000).cast("timestamp"))       # tiempo del evento (lectura)
           .withColumn("obs_ts", (F.col("obsTime") / 1000).cast("timestamp")))    # paso de 15 min del modelo
eventos.printSchema()

root
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTs: timestamp (nullable = true)
 |-- json_str: string (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- version: integer (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- estacion: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocidadViento: double (nullable = true)
 |-- direccionViento: double (nullable = true)
 |-- rafaga: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- presion: double (nullable = true)
 |-- obsTime: long (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- obs_ts: timestamp (nullable = true)



In [8]:
P_PRED = f"{ART}/predicciones_viento_s10"
CHK_PRED = f"{ART}/chk_s10_pred"

entrada = (eventos.filter((F.col("origen") == "open-meteo") & F.col("velocidadViento").isNotNull())
           .select("sensorId", "estacion", "obs_ts",
                   F.col("velocidadViento").alias("velocidad"), "rafaga",
                   F.col("direccionViento").alias("direccion"), "temperatura", "humedad", "presion"))

def predecir(df, batch_id):
    lote = features(df.dropDuplicates(["sensorId", "obs_ts"]).withColumn("ts_feat", F.col("obs_ts")))
    (modelo.transform(lote)
       .select(F.lit(batch_id).alias("batchId"), "sensorId", "estacion", F.col("obs_ts").alias("fechaHora"),
               "velocidad", "rafaga", F.round("prediction", 2).alias("prediccionVelocidad1h"),
               (F.col("obs_ts") + F.expr("INTERVAL 1 HOUR")).alias("fechaObjetivo"))
       .write.mode("append").parquet(P_PRED))

consulta_predicciones = (entrada.writeStream.foreachBatch(predecir)
                         .option("checkpointLocation", CHK_PRED).trigger(processingTime="10 seconds").start())

In [9]:
# esperar unos micro-lotes y ver las predicciones en vivo (se puede repetir)
time.sleep(50)
spark.read.parquet(P_PRED).orderBy(F.desc("batchId")).show(9, truncate=False)

+-------+-----------------+--------+-------------------+---------+------+---------------------+-------------------+
|batchId|sensorId         |estacion|fechaHora          |velocidad|rafaga|prediccionVelocidad1h|fechaObjetivo      |
+-------+-----------------+--------+-------------------+---------+------+---------------------+-------------------+
|3      |juliaca-viento-01|Juliaca |2026-11-02 20:15:00|5.2      |21.2  |5.69                 |2026-11-02 21:15:00|
|3      |lampa-viento-03  |Lampa   |2026-11-02 20:15:00|7.6      |18.0  |7.52                 |2026-11-02 21:15:00|
|3      |puno-viento-02   |Puno    |2026-11-02 20:15:00|4.3      |29.2  |7.23                 |2026-11-02 21:15:00|
|2      |juliaca-viento-01|Juliaca |2026-11-02 20:15:00|5.2      |21.2  |5.69                 |2026-11-02 21:15:00|
|2      |lampa-viento-03  |Lampa   |2026-11-02 20:15:00|7.6      |18.0  |7.52                 |2026-11-02 21:15:00|
|2      |puno-viento-02   |Puno    |2026-11-02 20:15:00|4.3      |29.2  

In [10]:
consulta_predicciones.stop()

**Hallazgo a reportar:** el modelo se entrena con ERA5 (reanálisis) y en vivo recibe el pronóstico
`current` de Open-Meteo (otro modelo). Puede haber sesgo entre ambos → por eso en S11 se mide el
**error en producción** y no solo el de test.